# 02 - Water mask, weak labels and chips
1. Build a common grid and a water mask.
2. Generate **weak** hyacinth labels from NDVI + FAI thresholds.
3. Cut chips and save an index.

The weak labels are only a starting point. Refine them (QGIS, or SAM-assisted with segment-geospatial, see https://samgeo.gishub.org) and drop refined rasters into `data/labels/refined/<scene_id>.tif` (1 = hyacinth). Any scene with a refined raster uses it instead of the threshold mask.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from hyacinthwatch.config import load_config
cfg = load_config("../config/config.yaml")
bbox = cfg["aoi"]["pilot_bbox"]
BANDS = cfg["bands"]
from pathlib import Path
from hyacinthwatch.stac_data import search_scenes, load_scene
from hyacinthwatch.indices import bands_dict, compute_indices
from hyacinthwatch.weak_labels import (valid_mask, permanent_water, polygon_mask,
                                       weak_hyacinth_mask, load_refined_mask)
from hyacinthwatch.chips import make_chips
from hyacinthwatch.inference import save_raster

In [ ]:
N_SCENES = 10
items = search_scenes(bbox, cfg["stac"]["date_range"], cfg["stac"]["max_cloud"])
items = items[:: max(len(items) // N_SCENES, 1)][:N_SCENES]
scenes, match = [], None
for it in items:
    stack, scl = load_scene(it, bbox, BANDS, match=match)
    match = match if match is not None else stack.isel(band=0, drop=True)
    scenes.append({"id": it.id, "date": it.datetime, "stack": stack, "valid": valid_mask(scl.values)})
    print(it.id, it.datetime.date())

In [ ]:
mn = np.stack([compute_indices(bands_dict(s["stack"].values, BANDS))["mndwi"] for s in scenes])
vs = np.stack([s["valid"] for s in scenes])
poly = cfg["aoi"]["water_polygon"]
water = polygon_mask(poly, scenes[0]["stack"]) if poly else permanent_water(mn, vs, cfg["weak_labels"]["water_min_freq"])
print("water fraction of AOI:", round(float(water.mean()), 3))
Path("../data/outputs").mkdir(exist_ok=True)
save_raster(water.astype("uint8"), scenes[0]["stack"], "../data/outputs/water_mask.tif", dtype="uint8")

In [ ]:
# Calibrate thresholds from the data, do not trust the defaults blindly
f = np.concatenate([compute_indices(bands_dict(s["stack"].values, BANDS))["fai"][water & s["valid"]] for s in scenes])
v = np.concatenate([compute_indices(bands_dict(s["stack"].values, BANDS))["ndvi"][water & s["valid"]] for s in scenes])
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(f, 200, range=(-0.05, 0.15)); ax[0].set_title("FAI over water")
ax[1].hist(v, 200, range=(-0.3, 0.8)); ax[1].set_title("NDVI over water")
plt.show()

In [ ]:
w = cfg["weak_labels"]; c = cfg["chips"]
frames = []
for s in scenes:
    idx = compute_indices(bands_dict(s["stack"].values, BANDS))
    mask = weak_hyacinth_mask(idx["ndvi"], idx["fai"], water, s["valid"], w["ndvi_min"], w["fai_min"])
    refined = Path(f"../data/labels/refined/{s['id']}.tif")
    if refined.exists():
        mask = load_refined_mask(refined, s["stack"])
    frames.append(make_chips(s["stack"].values, mask, s["valid"], water, s["stack"].rio.transform(),
                             s["id"], "../data/chips", c["size"], c["stride"], c["min_valid"], c["min_water"]))
index = pd.concat(frames, ignore_index=True)
index.to_csv("../data/chips/index.csv", index=False)
print(len(index), "chips;", int((index.pos_frac > 0).sum()), "contain hyacinth")